In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore' )
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)
print(f"Dataset shape: {df_food.shape}")
df_food.head()

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)


In [ ]:
# Task 2: Write your code here:
df_food.head()


In [ ]:
# Task 3: Write your code here:
df_food.info()


In [ ]:
# Task 4: Write your code here:
df_food.describe()



In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df_food, Delivery_Time):
  df_food[Delivery_Time].hist(bins=70, edgecolor='black')
  plt.title(f"Target Distribution ({Delivery_Time})")
  plt.xlabel(Delivery_Time)
  plt.ylabel("Frequency")
  plt.grid(False)
  plt.show()

check_target_distribution(df_food, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df_food.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df_food):
    missing_values = df_food.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df_food)

In [ ]:
missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
missing_data = pd.DataFrame({
'Column': missing_percentage.index,
'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] >
0].sort_values('Missing_Percentage', ascending=False)
print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
df_food.shape

In [ ]:
# 2. Do we have missing values?
def check_missing_values(df_food):
  missing_values = df_food.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_food)

In [ ]:
cols = ['Delivery_Time', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', "Vehicle_Type"]

df_clean = df_food[cols].copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_food.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Courier_Experience_yrs'])
print(f"After dropping missing Delivery_Time/Courier_Experience_yrs: {df_clean.shape}")

In [ ]:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
Categorical_cols = df_food.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(Categorical_cols))



In [ ]:
# Task 3: Write your code here:
for col in ['Weather', 'Traffic_Level', 'Time_of_Day',  "Vehicle_Type" ]:
  df_clean[col] = df_clean[col].fillna('unknown')
print("Missing values remaining:", df_clean.isnull().sum().sum())


In [ ]:
# Task 4: Write your code here:
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', "Vehicle_Type"]
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col].astype(str))
df_clean.head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import MinMaxScaler

features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = MinMaxScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df_food, Delivery_Time):
  print("Target Distribution:")
  print(df_food[Delivery_Time].value_counts(normalize=True))
  sns.countplot(x=df_food[ Delivery_Time])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_food, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
feature_cols =  [ 'Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', "Vehicle_Type"]
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")




In [ ]:
# Task 1: Write your code here:


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: